# 2D Explainer Shorts Studio (`Explainer_Shorts_Studio_6.ipynb`)

Automated episode tracking + resilient Edge-TTS retry/fallback + silent Flow clip audio + narration-synced Step cards + locked Pip character bible.

### What's New in v6
- **Resilient Edge-TTS (`NoAudioReceived` Auto-Retry & Voice Fallback):** Automatically retries transient Edge-TTS `NoAudioReceived` responses and rotates across fallback neural voices so Stage 1 doesn't crash on `scene_001`.
- **Safe Step Count Formatting:** Ensures `STEP_COUNT` is always a valid integer (never `None`) in script generation.
- **Automatic Episode Ledger & Queue (`episode_ledger.json`):** Tracks completed episodes under `MyDrive/GTA6_AutoClipper/explainer_2d/episode_ledger.json` and advances automatically when `RESUME_PROJECT_ID` is blank.


In [ ]:
#@title 2D Explainer Shorts Studio (Locked Character Bible + Auto-Synced Step Cards + Ken Burns + Word Pop Captions)
TOPIC = "auto" #@param {type:"string"}
FORMAT_TYPE = "YouTube Short (9:16)" #@param ["YouTube Short (9:16)", "Standard YouTube Video (16:9)"]
SCENE_COUNT = 12 #@param {type:"integer"}
CHARACTER_NAME = "Pip" #@param {type:"string"}
CHARACTER_BIBLE = "2D hand-drawn ink and soft watercolor cartoon on warm cream textured paper (#F6F1E7), minimalist round white-headed stick-figure character named Pip with simple black dot eyes, expressive eyebrows, wearing a signature muted terracotta hoodie and dark slate trousers, clean hand-inked black outlines, flat warm pastel accents, uncluttered background, no text or watermarks." #@param {type:"string"}
GENERATE_STEP_CARDS = "auto" #@param ["auto", "true", "false"]
STEP_COUNT = 3 #@param {type:"integer"}
VOICE = "en-US-AndrewMultilingualNeural" #@param ["en-US-AndrewMultilingualNeural", "en-US-BrianMultilingualNeural", "en-GB-RyanNeural", "en-US-AriaNeural"]
WHISPER_MODEL = "base.en" #@param ["tiny.en", "base.en", "small.en", "medium.en", "large-v3"]
WHISPER_DEVICE = "auto" #@param ["auto", "cuda", "cpu"]
MAX_SCENES_PER_FLOW_RUN = 8 #@param {type:"integer"}
RESUME_PROJECT_ID = "" #@param {type:"string"}
REPO_URL = "https://github.com/topboyasian-stack/gta6-autoclipper.git" #@param {type:"string"}
CODE_BRANCH = "improve/narrative-shorts-production" #@param {type:"string"}
ASSET_MODE = "manual_flow_or_stills" #@param ["manual_flow_or_stills", "gemini_image_api_if_missing", "veo_api_if_missing"]
IMAGE_MODEL = "imagen-3.0-generate-002" #@param {type:"string"}
VEO_MODEL = "veo-3.1-generate-preview" #@param {type:"string"}
CAPTION_STYLE = "word_pop_center" #@param ["word_pop_center", "lower_third_clean", "off"]
ENCODER_MODE = "auto" #@param ["auto", "nvenc", "x264"]
FLOW_SFX_VOLUME = 0.0 #@param {type:"number"}
VOICE_WARMTH_EQ = True #@param {type:"boolean"}

import hashlib
import html
import importlib.util
import json
import math
import os
import re
import shutil
import subprocess
import sys
import time
import traceback
from pathlib import Path
from google.colab import drive, userdata
from IPython.display import HTML, Markdown, display

ROOT = Path("/content/gta6-autoclipper")

PINNED_PKGS = [
    "google-genai==2.12.1",
    "pydantic==2.10.6",
    "python-dotenv==1.0.1",
    "edge-tts==7.0.2",
    "faster-whisper==1.1.1",
    "av==17.1.0",
]

REQUIRED_MODULES = [
    "pydantic",
    "dotenv",
    "edge_tts",
    "faster_whisper",
    "av",
    "PIL",
]

PRE_BLOCK_STYLE = (
    "white-space:pre-wrap; user-select:all; margin-top:8px; "
    "padding:10px; background:#1e1e1e; color:#f0f0f0; border-radius:4px;"
)
DETAILS_STYLE = "margin-top:12px; border:1px solid #444; border-radius:6px; padding:10px;"

SERIES_QUEUE = [
    "Why getting a raise still leaves most people broke: lifestyle creep and the leaky bucket trap.",
    "Why saving money feels harder than spending it: the psychology of instant rewards and future goals.",
    "Why buying things to look rich can keep you financially stuck: status signals versus financial security.",
    "Why good habits fail when motivation disappears: how your environment shapes repeated behaviour.",
    "Why small subscriptions feel harmless but quietly drain a budget: the hidden cost of recurring choices.",
    "Why people keep investing effort in a losing plan: the sunk-cost trap in everyday decisions.",
    "Why checking notifications fragments your attention: the reward loop behind constant checking.",
    "Why starting small beats waiting for the perfect plan: making progress easier to repeat.",
    "Why a bigger paycheck can still feel like no progress: lifestyle inflation explained visually.",
    "Why people overspend to match their friends: social comparison and money decisions.",
]


def normalize_topic(text: str) -> str:
    return re.sub(r"[^a-z0-9]+", " ", str(text or "").lower()).strip()


def topic_terms(text: str) -> set:
    stop = {
        "why", "what", "when", "where", "which", "how", "the", "and", "for", "with", "from",
        "into", "that", "this", "your", "you", "are", "can", "will", "still", "most", "people",
        "explained", "simply", "simple", "like", "five", "everyday", "about", "behind", "their",
        "they", "them", "than", "because", "actually", "feels", "feel", "keep", "keeps", "quietly",
        "makes", "made", "using", "used", "gets", "getting", "most", "every", "over", "under",
    }
    return {w for w in normalize_topic(text).split() if len(w) >= 4 and w not in stop}


def topics_are_similar(a: str, b: str) -> bool:
    left, right = topic_terms(a), topic_terms(b)
    if not left or not right:
        return normalize_topic(a) == normalize_topic(b)
    overlap = len(left & right)
    return (overlap / max(1, len(left)) >= 0.65) or (overlap / max(1, len(left | right)) >= 0.55)


def topic_for_episode(ep_num: int) -> str:
    if ep_num <= 1:
        return "How compound interest quietly turns tiny daily habits into massive wealth."
    queue_index = ep_num - 2
    if 0 <= queue_index < len(SERIES_QUEUE):
        return SERIES_QUEUE[queue_index]
    return "An everyday money or behaviour pattern most people overlook: why it happens and one useful takeaway."


def episode_plan_matches_topic(episode: dict, target_topic: str) -> bool:
    fields = [
        episode.get("topic", ""),
        episode.get("title", ""),
        episode.get("logline", ""),
        episode.get("synopsis", ""),
        episode.get("narration_script", ""),
    ]
    scenes = episode.get("scenes", [])
    if isinstance(scenes, list):
        for scene in scenes:
            if isinstance(scene, dict):
                fields.extend(scene.get(k, "") for k in ("title", "scene_title", "narration", "text"))
    saved_text = normalize_topic(" ".join(str(item or "") for item in fields))
    target_norm = normalize_topic(target_topic)
    if not target_norm or not saved_text:
        return False
    if target_norm in saved_text:
        return True
    target_terms = topic_terms(target_topic)
    saved_terms = set(saved_text.split())
    if not target_terms:
        return False
    return len(target_terms & saved_terms) / len(target_terms) >= 0.35


def generate_next_topic_with_gemini(api_key: str, ep_num: int, ledger: dict) -> str:
    if not api_key:
        raise RuntimeError("No Gemini API key is available for topic generation.")
    from google import genai
    from google.genai import types
    try:
        from config import PLANNER_MODEL
        model_name = PLANNER_MODEL
    except Exception:
        model_name = "gemini-2.5-flash"
    used = [str(item.get("topic") or "") for item in ledger.get("episodes", [])]
    prompt = (
        "Generate ONE fresh topic premise for the YouTube channel Pip & The Pattern. "
        "The channel publishes original hand-drawn explainers about the hidden patterns behind money, "
        "habits, and human behaviour. Each topic must teach one concrete insight, be visually explainable "
        "in 60 seconds, have a specific curiosity hook, and lead to a useful takeaway. Avoid generic "
        "motivation, unsupported financial promises, copied titles, and forced numbered steps. "
        f"This is episode {ep_num}. Avoid these used episode topics: {json.dumps(used, ensure_ascii=False)}. "
        "Return only the topic premise, no numbering, no quotation marks, and no extra commentary."
    )
    client = genai.Client(api_key=api_key)
    response = client.models.generate_content(
        model=model_name,
        contents=prompt,
        config=types.GenerateContentConfig(temperature=0.75),
    )
    candidate = str(getattr(response, "text", "") or "").strip().strip("\"'").splitlines()[0].strip()
    if len(candidate) < 18 or len(candidate) > 220:
        raise RuntimeError("Gemini returned an unusable topic premise.")
    if any(topics_are_similar(candidate, used_topic) for used_topic in used if used_topic.strip()):
        raise RuntimeError("Gemini returned a topic too similar to an existing episode.")
    approved_terms = {
        "money", "saving", "spending", "spend", "salary", "paycheck", "budget", "debt", "wealth",
        "rich", "poor", "habit", "habits", "behaviour", "behavior", "psychology", "brain", "focus",
        "attention", "discipline", "motivation", "decision", "decisions", "bias", "reward", "rewards",
        "procrastination", "status", "comparison", "routine", "consistency", "impulse", "risk", "time",
        "subscription", "subscriptions", "investing", "invest", "learning", "memory", "stress", "spending",
    }
    if not (topic_terms(candidate) & approved_terms):
        raise RuntimeError("Gemini topic did not clearly fit one of the channel's content pillars.")
    return candidate


def recover_unlogged_projects(ledger: dict, roots: list) -> None:
    episodes = ledger.setdefault("episodes", [])
    known_ids = {str(item.get("project_id") or "") for item in episodes}
    ep1_id = "20261126_223959_explainer_2d_How_compound_interest_quietly_turns_tiny_dail"
    for root in roots:
        root = Path(root)
        if not root.is_dir():
            continue
        for project_root in root.iterdir():
            if not project_root.is_dir() or project_root.name in known_ids:
                continue
            project_id = project_root.name
            plan_path = project_root / "explainer_2d" / "episode.json"
            if not plan_path.is_file():
                continue
            match = re.search(r"(?:^|_)EP(\d{2})_", project_id, flags=re.IGNORECASE)
            if project_id == ep1_id:
                ep_num = 1
                topic = topic_for_episode(1)
            elif match:
                ep_num = int(match.group(1))
                topic = topic_for_episode(ep_num)
            else:
                continue
            final_path = project_root / "explainer_2d" / "assembled" / "final_explainer_short.mp4"
            status = "completed" if final_path.is_file() else "awaiting_visuals"
            episodes.append({
                "episode_number": ep_num,
                "project_id": project_id,
                "topic": topic,
                "status": status,
                "updated_at": time.strftime("%Y-%m-%d %H:%M:%S"),
                "recovered_from_project_folder": True,
            })
            known_ids.add(project_id)


def refresh_ledger_cursor(ledger: dict) -> None:
    episodes = ledger.get("episodes", [])
    pending = [
        item for item in episodes
        if str(item.get("status") or "").lower() not in {"completed", "abandoned", "skipped"}
        and item.get("project_id") and item.get("topic")
    ]
    if pending:
        ledger["next_episode_number"] = min(int(item.get("episode_number", 1)) for item in pending)
    else:
        completed_numbers = [
            int(item.get("episode_number", 0)) for item in episodes
            if str(item.get("status") or "").lower() == "completed"
        ]
        ledger["next_episode_number"] = max(completed_numbers + [0]) + 1


def load_or_init_ledger(ledger_path: Path) -> dict:
    if ledger_path.is_file():
        try:
            data = json.loads(ledger_path.read_text(encoding="utf-8"))
            if isinstance(data, dict) and isinstance(data.get("episodes"), list):
                return data
        except Exception:
            pass
    seeded = []
    ep1_id = "20261126_223959_explainer_2d_How_compound_interest_quietly_turns_tiny_dail"
    ep1_topic = "How compound interest quietly turns tiny daily habits into massive wealth, explained like you're five."
    ep1_found = (
        (Path("/content/drive/MyDrive/GTA6_AutoClipper/projects") / ep1_id / "explainer_2d" / "assembled" / "final_explainer_short.mp4").is_file()
        or (Path("/content/work_projects") / ep1_id / "explainer_2d" / "assembled" / "final_explainer_short.mp4").is_file()
    )
    if ep1_found:
        seeded.append({
            "episode_number": 1,
            "project_id": ep1_id,
            "topic": ep1_topic,
            "status": "completed",
            "updated_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        })
    return {
        "series_name": "Pip & The Pattern",
        "next_episode_number": len(seeded) + 1,
        "episodes": seeded,
    }


def save_ledger(ledger_path: Path, ledger: dict):
    ledger_path.parent.mkdir(parents=True, exist_ok=True)
    ledger_path.write_text(json.dumps(ledger, indent=2), encoding="utf-8")


def infer_topic_from_project_id(project_id: str, ep_num: int, existing_topic: str = "") -> str:
    marker = f"EP{int(ep_num):02d}_"
    project_text = str(project_id or "")
    marker_pos = project_text.upper().find(marker.upper())
    if marker_pos >= 0:
        slug = normalize_topic(project_text[marker_pos + len(marker):])
        if slug:
            for candidate in SERIES_QUEUE:
                expected = normalize_topic(candidate)
                prefix_len = min(len(slug), len(expected))
                if prefix_len >= 25 and expected.startswith(slug[:prefix_len - min(3, max(0, prefix_len - 25))]):
                    return candidate
                if prefix_len >= 25 and slug.startswith(expected[:prefix_len - min(3, max(0, prefix_len - 25))]):
                    return candidate
    return existing_topic.strip() or topic_for_episode(ep_num)


def resolve_episode_plan(topic_input: str, resume_id: str, ledger: dict):
    cleaned_resume = str(resume_id or "").strip()
    episodes = ledger.setdefault("episodes", [])

    # Repair stale topic labels from known project IDs before selecting any pending episode.
    for ep in episodes:
        project_id = str(ep.get("project_id") or "")
        try:
            ep_num = int(ep.get("episode_number", 0))
        except Exception:
            ep_num = 0
        if not project_id or ep_num <= 0:
            continue
        inferred = infer_topic_from_project_id(project_id, ep_num, str(ep.get("topic") or ""))
        if inferred and not topics_are_similar(str(ep.get("topic") or ""), inferred):
            print(
                f"Correcting stale ledger topic for {project_id}: "
                f"{ep.get('topic')!r} -> {inferred!r}",
                flush=True,
            )
            ep["topic"] = inferred
            ep["updated_at"] = time.strftime("%Y-%m-%d %H:%M:%S")

    if cleaned_resume and cleaned_resume.lower() != "insert text here":
        for ep in episodes:
            if ep.get("project_id") == cleaned_resume:
                return int(ep.get("episode_number", 1)), str(ep.get("topic") or topic_for_episode(1)).strip(), cleaned_resume
        match = re.search(r"EP(\d{2})", cleaned_resume, flags=re.IGNORECASE)
        if not match:
            raise ValueError(
                "RESUME_PROJECT_ID is not in the ledger and has no EP## marker. Clear it to resume the next pending episode, "
                "or use the exact project ID printed by the studio."
            )
        ep_num = int(match.group(1))
        chosen_topic = infer_topic_from_project_id(cleaned_resume, ep_num, topic_for_episode(ep_num))
        record_ledger_entry(ledger, ep_num, cleaned_resume, chosen_topic, "planned")
        return ep_num, chosen_topic, cleaned_resume

    refresh_ledger_cursor(ledger)
    pending = sorted(
        (
            item for item in episodes
            if str(item.get("status") or "").lower() not in {"completed", "abandoned", "skipped"}
            and item.get("project_id") and item.get("topic")
        ),
        key=lambda item: int(item.get("episode_number", 1)),
    )
    if pending:
        item = pending[0]
        return int(item.get("episode_number", 1)), str(item["topic"]).strip(), str(item["project_id"]).strip()

    ep_num = int(ledger.get("next_episode_number", len(episodes) + 1))
    raw_topic = str(topic_input or "").strip()
    used_topics = [str(item.get("topic") or "") for item in episodes]
    if raw_topic and raw_topic.lower() not in {"auto", "generate", "next"}:
        if not any(topics_are_similar(raw_topic, used) for used in used_topics if used.strip()):
            chosen_topic = raw_topic
        else:
            print("Requested topic matches an existing episode; choosing a fresh channel-fit topic instead.", flush=True)
            raw_topic = "auto"
    else:
        raw_topic = "auto"
    if raw_topic == "auto":
        try:
            chosen_topic = generate_next_topic_with_gemini(TOPIC_API_KEY, ep_num, ledger)
            print(f"AI topic selected for Episode #{ep_num:02d}: {chosen_topic}", flush=True)
        except Exception as topic_exc:
            chosen_topic = next(
                (item for item in SERIES_QUEUE
                 if not any(topics_are_similar(item, used) for used in used_topics if used.strip())),
                topic_for_episode(ep_num),
            )
            print(
                f"AI topic generation unavailable ({type(topic_exc).__name__}: {topic_exc}); "
                f"using curated channel-fit fallback: {chosen_topic}",
                flush=True,
            )
    proj_id = make_project_id(f"EP{ep_num:02d}_{chosen_topic}", "")
    record_ledger_entry(ledger, ep_num, proj_id, chosen_topic, "planned")
    return ep_num, chosen_topic, proj_id


def record_ledger_entry(ledger: dict, ep_num: int, proj_id: str, chosen_topic: str, status: str):
    episodes = ledger.get("episodes", [])
    existing = next((item for item in episodes if item.get("project_id") == proj_id), None)
    entry = {
        "episode_number": ep_num,
        "project_id": proj_id,
        "topic": chosen_topic,
        "status": status,
        "updated_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    }
    if existing:
        existing.update(entry)
    else:
        episodes.append(entry)
    episodes.sort(key=lambda x: int(x.get("episode_number", 0)))
    ledger["episodes"] = episodes
    completed_count = sum(1 for item in episodes if item.get("status") == "completed")
    max_ep = max([int(item.get("episode_number", 0)) for item in episodes] + [0])
    if status == "completed":
        ledger["next_episode_number"] = max(max_ep + 1, completed_count + 1)
    else:
        ledger["next_episode_number"] = max(ep_num, max_ep if max_ep >= ep_num else ep_num)


def make_project_id(topic_text, resume_id):
    cleaned_resume = str(resume_id or "").strip()
    if cleaned_resume and cleaned_resume.lower() != "insert text here":
        return cleaned_resume
    slug = re.sub(r"[^a-zA-Z0-9]+", "_", str(topic_text or "")).strip("_")[:45] or "explainer"
    return f"{time.strftime('%Y%m%d_%H%M%S')}_explainer_2d_{slug}"


def resolve_whisper_device(requested):
    if requested in ("cuda", "cpu"):
        return requested
    try:
        res = subprocess.run(["nvidia-smi"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        if res.returncode == 0:
            return "cuda"
    except Exception:
        pass
    return "cpu"


def should_copy_file(src: Path, dst: Path) -> bool:
    if not dst.is_file():
        return True
    s_stat = src.stat()
    d_stat = dst.stat()
    return s_stat.st_size != d_stat.st_size or int(s_stat.st_mtime) > int(d_stat.st_mtime) + 1


def sync_tree(src_dir, dst_dir, skip_patterns=("episode.backup.",)):
    src_path = Path(src_dir)
    dst_path = Path(dst_dir)
    if not src_path.is_dir():
        return 0
    dst_path.mkdir(parents=True, exist_ok=True)
    copied = 0
    for item in src_path.rglob("*"):
        if any(pat in item.name for pat in skip_patterns):
            continue
        rel = item.relative_to(src_path)
        target = dst_path / rel
        if item.is_dir():
            target.mkdir(parents=True, exist_ok=True)
        elif item.is_file() and should_copy_file(item, target):
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(item, target)
            copied += 1
    return copied


ACTIVE_WHISPER_DEVICE = resolve_whisper_device(WHISPER_DEVICE)
IS_SHORT = "Short" in FORMAT_TYPE
TARGET_W, TARGET_H = (1080, 1920) if IS_SHORT else (1920, 1080)
ASPECT_STR = "9:16" if IS_SHORT else "16:9"
FINAL_VIDEO_NAME = "final_explainer_short.mp4" if IS_SHORT else "final_explainer_16x9.mp4"

if not Path("/content/drive/MyDrive").is_dir():
    print("Connecting Google Drive…", flush=True)
    drive.mount("/content/drive")

try:
    TOPIC_API_KEY = str(userdata.get("GEMINI_API_KEY") or "").strip()
except Exception:
    TOPIC_API_KEY = str(os.environ.get("GEMINI_API_KEY") or "").strip()
if len(TOPIC_API_KEY) < 10:
    TOPIC_API_KEY = ""

DRIVE_WORKSPACE = Path("/content/drive/MyDrive/GTA6_AutoClipper")
SERIES_DIR = DRIVE_WORKSPACE / "explainer_2d"
SERIES_DIR.mkdir(parents=True, exist_ok=True)
LEDGER_PATH = SERIES_DIR / "episode_ledger.json"
LEDGER = load_or_init_ledger(LEDGER_PATH)
recover_unlogged_projects(
    LEDGER,
    [
        DRIVE_WORKSPACE / "projects",
        Path("/content/work_projects"),
    ],
)
refresh_ledger_cursor(LEDGER)
EPISODE_NUM, ACTIVE_TOPIC, PROJECT_ID = resolve_episode_plan(TOPIC, RESUME_PROJECT_ID, LEDGER)
TOPIC = ACTIVE_TOPIC
save_ledger(LEDGER_PATH, LEDGER)

DRIVE_PROJECT_DIR = DRIVE_WORKSPACE / "projects" / PROJECT_ID
DRIVE_OUTPUT_DIR = DRIVE_PROJECT_DIR / "explainer_2d"
LOCAL_PROJECT_DIR = Path("/content/work_projects") / PROJECT_ID
LOCAL_OUTPUT_DIR = LOCAL_PROJECT_DIR / "explainer_2d"
LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CLIPS_DIR = LOCAL_OUTPUT_DIR / "flow_clips"
CLIPS_DIR.mkdir(parents=True, exist_ok=True)
FRAMES_DIR = LOCAL_OUTPUT_DIR / "frames"
FRAMES_DIR.mkdir(parents=True, exist_ok=True)
ASSEMBLED_DIR = LOCAL_OUTPUT_DIR / "assembled"
ASSEMBLED_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = LOCAL_OUTPUT_DIR / "explainer_run.log"

log_stream = LOG_PATH.open("a", encoding="utf-8")


def flush_log():
    try:
        log_stream.flush()
    except Exception:
        pass


def write_log(text, flush=False):
    log_stream.write(text)
    if flush:
        flush_log()


def note(message):
    stamp = time.strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{stamp}] {message}"
    print(line, flush=True)
    write_log(line + "\n", flush=True)


def run_cmd(command, env=None, check=True):
    proc = subprocess.run(
        [str(p) for p in command],
        cwd=str(ROOT) if ROOT.is_dir() else None,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        errors="replace",
    )
    if check and proc.returncode != 0:
        raise RuntimeError(
            f"Command failed ({proc.returncode}): {' '.join(map(str, command))}\n{proc.stdout[-1500:]}"
        )
    return proc


def run_logged(label, command, env=None):
    note("START: " + label)
    write_log("COMMAND: " + " ".join(str(part) for part in command) + "\n", flush=True)
    process = subprocess.Popen(
        [str(part) for part in command],
        cwd=str(ROOT) if ROOT.is_dir() else None,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        errors="replace",
    )
    write_log("PROCESS_ID: " + str(process.pid) + "\n", flush=True)
    print("Process started: " + str(process.pid), flush=True)

    line_count = 0
    for line in process.stdout:
        write_log(line, flush=False)
        print(line, end="", flush=True)
        line_count += 1
        if line_count % 25 == 0:
            flush_log()

    return_code = process.wait()
    write_log("EXIT_CODE: " + str(return_code) + "\n", flush=True)
    flush_log()
    if return_code != 0:
        raise RuntimeError(f"{label} failed with exit code {return_code}. See {LOG_PATH}")
    note("DONE: " + label)


def ensure_system_tools():
    if shutil.which("ffmpeg") and shutil.which("ffprobe") and Path("/content/.sys_tools_ready").is_file():
        note("FFmpeg, ffprobe, and system fonts are already available.")
        return
    run_logged(
        "Install FFmpeg and fonts",
        ["bash", "-lc", "apt-get update -qq && apt-get install -y -qq ffmpeg fonts-dejavu-core fonts-dejavu-extra"],
    )
    Path("/content/.sys_tools_ready").touch()


def ensure_python_deps(req_file: Path):
    if req_file.is_file():
        raw_lines = req_file.read_text(encoding="utf-8", errors="replace").splitlines()
        filtered_lines = [
            ln for ln in raw_lines
            if ln.strip() and not re.match(r"^\s*(pillow|pil)\b", ln, flags=re.IGNORECASE)
        ]
        safe_req = Path("/content/.requirements_no_pillow.txt")
        safe_req.write_text("\n".join(filtered_lines) + "\n", encoding="utf-8")
        spec_bytes = safe_req.read_bytes()
        cmd = [sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", "-r", str(safe_req)]
    else:
        spec_bytes = "\n".join(PINNED_PKGS).encode("utf-8")
        cmd = [sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", *PINNED_PKGS]

    req_hash = hashlib.sha256(spec_bytes).hexdigest()[:16]
    marker = Path(f"/content/.deps_installed_{req_hash}")
    if marker.is_file() and all(importlib.util.find_spec(m) is not None for m in REQUIRED_MODULES):
        note("Python dependencies already satisfied for current spec; skipping pip install.")
        return

    run_logged("Install Python dependencies", cmd)
    marker.touch()


def probe_media_duration(path: Path) -> float:
    out = run_cmd([
        "ffprobe", "-v", "error", "-show_entries", "format=duration",
        "-of", "default=noprint_wrappers=1:nokey=1", str(path),
    ]).stdout.strip()
    return max(0.1, float(out))


def has_audio_stream(path: Path) -> bool:
    out = run_cmd([
        "ffprobe", "-v", "error", "-select_streams", "a",
        "-show_entries", "stream=codec_type", "-of", "csv=p=0", str(path),
    ], check=False).stdout.strip()
    return "audio" in out


def detect_video_encoder(mode: str) -> str:
    if mode == "x264":
        return "libx264"
    test_cmd = [
        "ffmpeg", "-y", "-f", "lavfi", "-i", "color=c=black:s=256x256:d=0.1",
        "-c:v", "h264_nvenc", "-f", "null", "-",
    ]
    res = subprocess.run(test_cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    if res.returncode == 0:
        return "h264_nvenc"
    if mode == "nvenc":
        raise RuntimeError("Requested h264_nvenc, but NVENC probe failed on this runtime.")
    return "libx264"


def render_handdrawn_step_card(step_num: int, step_title: str, out_png: Path, width: int, height: int):
    out_png.parent.mkdir(parents=True, exist_ok=True)
    font_candidates = [
        "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
        "/usr/share/fonts/truetype/dejavu/DejaVuSerif-Bold.ttf",
    ]
    font_path = next((p for p in font_candidates if Path(p).is_file()), "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf")

    big_size = int(min(width, height) * 0.105)
    sub_size = int(min(width, height) * 0.042)
    margin = int(min(width, height) * 0.08)
    inner_margin = margin + 10

    badge_w = int(width * 0.54)
    badge_h = int(big_size * 1.65)
    badge_x = (width - badge_w) // 2
    badge_y = int(height * 0.37)

    clean_sub = re.sub(r"\s+", " ", str(step_title or "")).strip()
    clean_sub = re.sub(r"^step\s*(\d+|one|two|three|four|five)\s*[:\-–—]\s*", "", clean_sub, flags=re.IGNORECASE).strip()
    if len(clean_sub) > 42:
        clean_sub = clean_sub[:39].rstrip() + "..."

    def esc_drawtext(txt: str) -> str:
        return (
            txt.replace("\\", "\\\\")
            .replace(":", "\\:")
            .replace("'", "\\'")
            .replace("%", "\\%")
            .replace(",", "\\,")
        )

    step_label = esc_drawtext(f"Step {step_num}")
    sub_label = esc_drawtext(clean_sub)

    filters = [
        f"drawbox=x={margin}:y={margin}:w={width - 2 * margin}:h={height - 2 * margin}:color=#26231F:t=5",
        f"drawbox=x={inner_margin}:y={inner_margin}:w={width - 2 * inner_margin}:h={height - 2 * inner_margin}:color=#26231F:t=2",
        f"drawbox=x={width // 2 - 26}:y={badge_y - 72}:w=52:h=4:color=#26231F:t=fill",
        f"drawbox=x={width // 2 - 2}:y={badge_y - 96}:w=4:h=52:color=#26231F:t=fill",
        f"drawbox=x={badge_x}:y={badge_y}:w={badge_w}:h={badge_h}:color=#E58B6B:t=fill",
        f"drawbox=x={badge_x}:y={badge_y}:w={badge_w}:h={badge_h}:color=#26231F:t=5",
        f"drawtext=fontfile='{font_path}':text='{step_label}':fontcolor=#1E1B18:fontsize={big_size}:"
        f"x=(w-text_w)/2:y={badge_y}+({badge_h}-text_h)/2",
    ]
    if sub_label:
        filters.append(
            f"drawtext=fontfile='{font_path}':text='{sub_label}':fontcolor=#3B352E:fontsize={sub_size}:"
            f"x=(w-text_w)/2:y={badge_y + badge_h + 52}"
        )

    run_cmd([
        "ffmpeg", "-y",
        "-f", "lavfi", "-i", f"color=c=#F6F1E7:s={width}x{height}:d=0.1",
        "-vf", ",".join(filters),
        "-frames:v", "1",
        str(out_png),
    ])
    return out_png


def should_use_step_cards(mode_setting, ep_num: int, topic_text: str, scenes: list) -> bool:
    if isinstance(mode_setting, bool):
        return mode_setting
    norm_mode = str(mode_setting or "auto").strip().lower()
    if norm_mode in ("true", "yes", "on", "1"):
        return True
    if norm_mode in ("false", "no", "off", "0"):
        return False

    # Auto mode requires both an explicit list/sequence promise and explicit step labels in the plan.
    topic_blob = str(topic_text or "").lower()
    list_promise = re.search(
        r"\b(?:step[- ]by[- ]step|\d+\s+(?:steps|ways|rules|mistakes|lessons|habits)|"
        r"(?:three|four|five|six|seven|eight|nine|ten)\s+(?:steps|ways|rules|mistakes|lessons|habits))\b",
        topic_blob,
    )
    if not list_promise:
        return False
    explicit_step_scenes = 0
    for sc in scenes or []:
        scene_text = " ".join(
            str(sc.get(k) or "") for k in ("title", "scene_title", "narration", "text")
        )
        if re.search(r"\bstep\s*(?:\d+|one|two|three|four|five|six|seven|eight|nine|ten)\b", scene_text, re.I):
            explicit_step_scenes += 1
    return explicit_step_scenes >= 2


def compute_step_card_slots(scenes: list, step_count: int, ep_num: int = 1, topic_text: str = "") -> dict:
    scene_count = len(scenes)
    if step_count <= 0 or scene_count < 6:
        return {}
    if not should_use_step_cards(GENERATE_STEP_CARDS, ep_num, topic_text, scenes):
        return {}

    ordinal_map = {
        1: ("1", "one", "first"),
        2: ("2", "two", "second"),
        3: ("3", "three", "third"),
        4: ("4", "four", "fourth"),
        5: ("5", "five", "fifth"),
        6: ("6", "six", "sixth"),
    }
    matched_slots = {}
    used_indices = set()
    for step_num in range(1, step_count + 1):
        tokens = ordinal_map.get(step_num, (str(step_num),))
        pat = re.compile(r"\bstep\s*(?:" + "|".join(re.escape(t) for t in tokens) + r")\b", flags=re.I)
        for idx, sc in enumerate(scenes, start=1):
            if idx in used_indices:
                continue
            scene_text = " ".join(str(sc.get(k) or "") for k in ("title", "scene_title", "narration", "text"))
            if pat.search(scene_text):
                matched_slots[idx] = step_num
                used_indices.add(idx)
                break
    if str(GENERATE_STEP_CARDS or "auto").strip().lower() == "auto":
        return dict(sorted(matched_slots.items()))
    if not matched_slots:
        usable_start = 2
        usable_end = max(usable_start + 1, scene_count - 2)
        span = usable_end - usable_start
        for step_num in range(1, step_count + 1):
            idx = usable_start + round((step_num - 1) * span / max(1, step_count - 1)) if step_count > 1 else usable_start
            while idx in matched_slots and idx < scene_count - 1:
                idx += 1
            matched_slots[idx] = step_num
    return dict(sorted(matched_slots.items()))


def find_scene_asset(clips_dir: Path, frames_dir: Path, scene_idx: int):
    for ext in (".mp4", ".mov", ".webm", ".mkv"):
        for prefix in (f"scene_{scene_idx:02d}", f"scene_{scene_idx:03d}"):
            cand = clips_dir / f"{prefix}{ext}"
            if cand.is_file() and cand.stat().st_size > 0:
                return ("video", cand)
    for ext in (".png", ".jpg", ".jpeg", ".webp"):
        for prefix in (f"scene_{scene_idx:02d}", f"scene_{scene_idx:03d}"):
            cand = frames_dir / f"{prefix}{ext}"
            if cand.is_file() and cand.stat().st_size > 0:
                return ("image", cand)
    return (None, None)


def try_generate_still_with_imagen(api_key: str, model_name: str, full_prompt: str, out_png: Path, aspect: str):
    if not api_key or importlib.util.find_spec("google.genai") is None:
        return False
    from google import genai
    from google.genai import types

    client = genai.Client(api_key=api_key)
    note(f"Generating 2D illustration via {model_name} for {out_png.name}...")
    res = client.models.generate_images(
        model=model_name,
        prompt=full_prompt,
        config=types.GenerateImagesConfig(number_of_images=1, aspect_ratio=aspect),
    )
    imgs = getattr(res, "generated_images", None) or []
    if not imgs:
        return False
    out_png.parent.mkdir(parents=True, exist_ok=True)
    imgs[0].image.save(str(out_png))
    return out_png.is_file() and out_png.stat().st_size > 0


def try_generate_clip_with_veo(api_key: str, model_name: str, full_prompt: str, out_mp4: Path, aspect: str):
    if not api_key or importlib.util.find_spec("google.genai") is None:
        return False
    from google import genai
    from google.genai import types

    client = genai.Client(api_key=api_key)
    note(f"Generating 2D animated clip via {model_name} for {out_mp4.name}...")
    op = client.models.generate_videos(
        model=model_name,
        prompt=full_prompt,
        config=types.GenerateVideosConfig(aspect_ratio=aspect),
    )
    deadline = time.time() + 300
    while not op.done:
        if time.time() > deadline:
            raise TimeoutError(f"Timed out waiting for Veo generation on {out_mp4.name}")
        time.sleep(5)
        op = client.operations.get(op)
    generated = getattr(op.response, "generated_videos", None) or []
    if not generated:
        return False
    video_obj = generated[0].video
    client.files.download(file=video_obj)
    video_obj.save(str(out_mp4))
    return out_mp4.is_file() and out_mp4.stat().st_size > 0


def build_locked_explainer_prompts(scenes: list, step_slots: dict, out_dir: Path):
    lines = [
        "# 2D Explainer Shorts — Locked Character & Art-Style Bible Prompts",
        "",
        f"**Topic:** {TOPIC}",
        f"**Recurring Character:** {CHARACTER_NAME}",
        f"**Aspect Ratio:** {ASPECT_STR} (`{TARGET_W}x{TARGET_H}`)",
        "",
        "## Locked Character & Art-Style Bible (Included in Every Scene Below)",
        f"> {CHARACTER_BIBLE}",
        "",
        "## Single-Click Copy-Paste Prompts for Google Flow / Image Generator",
        "",
    ]
    manifest_rows = []
    for idx, sc in enumerate(scenes, start=1):
        action = str(
            sc.get("visual_prompt")
            or sc.get("image_prompt")
            or sc.get("prompt")
            or sc.get("narration")
            or sc.get("text")
            or ""
        ).strip()
        narration_preview = str(sc.get("narration") or sc.get("text") or "").strip()

        if idx in step_slots:
            step_num = step_slots[idx]
            lines.extend([
                f"### Scene {idx:02d} — AUTO-GENERATED STEP CARD (`Step {step_num}`)",
                f"- **Status:** Already rendered automatically as `frames/scene_{idx:02d}.png` (no Flow generation needed).",
                f"- **Narration:** {narration_preview}",
                "",
            ])
            continue

        locked_prompt = (
            f"{CHARACTER_BIBLE} "
            f"Scene action featuring {CHARACTER_NAME}: {action}. "
            f"Aspect ratio {ASPECT_STR}, smooth subtle 2D animation, clean composition with clear center space for captions. "
            f"No text, no speech bubbles, no subtitles, no watermarks."
        )
        lines.extend([
            f"### Scene {idx:02d} — Save video as `scene_{idx:02d}.mp4` (or still image as `scene_{idx:02d}.png`)",
            f"- **Narration:** {narration_preview}",
            "```text",
            locked_prompt,
            "```",
            "",
        ])
        manifest_rows.append((idx, locked_prompt))

    md_content = "\n".join(lines)
    (out_dir / "explainer_locked_prompts.md").write_text(md_content, encoding="utf-8")
    return md_content, dict(manifest_rows)


def generate_word_pop_ass(audio_tracks: list, whisper_model_name: str, device: str, out_ass: Path, width: int, height: int):
    from faster_whisper import WhisperModel

    compute_type = "float16" if device == "cuda" else "int8"
    model = WhisperModel(whisper_model_name, device=device, compute_type=compute_type)

    def fmt_ass_time(sec: float) -> str:
        sec = max(0.0, sec)
        h = int(sec // 3600)
        m = int((sec % 3600) // 60)
        s = int(sec % 60)
        cs = int(round((sec - math.floor(sec)) * 100))
        if cs >= 100:
            s += 1
            cs = 0
        return f"{h}:{m:02d}:{s:02d}.{cs:02d}"

    font_sz = 64 if width < height else 58
    header = [
        "[Script Info]",
        "ScriptType: v4.00+",
        f"PlayResX: {width}",
        f"PlayResY: {height}",
        "WrapStyle: 2",
        "ScaledBorderAndShadow: yes",
        "",
        "[V4+ Styles]",
        "Format: Name, Fontname, Fontsize, PrimaryColour, SecondaryColour, OutlineColour, BackColour, "
        "Bold, Italic, Underline, StrikeOut, ScaleX, ScaleY, Spacing, Angle, BorderStyle, Outline, Shadow, "
        "Alignment, MarginL, MarginR, MarginV, Encoding",
        f"Style: WordPop,DejaVu Sans,{font_sz},&H00FFFFFF,&H00FFFFFF,&H00181818,&H80000000,"
        f"-1,0,0,0,100,100,0,0,1,4,2,5,60,60,60,1",
        "",
        "[Events]",
        "Format: Layer, Start, End, Style, Name, MarginL, MarginR, MarginV, Effect, Text",
    ]

    events = []
    cursor = 0.0
    for wav_path, slot_dur in audio_tracks:
        segments, _ = model.transcribe(str(wav_path), word_timestamps=True, language="en")
        words = []
        for seg in segments:
            for w in getattr(seg, "words", None) or []:
                token = str(w.word or "").strip()
                if token:
                    words.append((float(w.start), float(w.end), token))

        for i, (ws, we, token) in enumerate(words):
            start_t = cursor + ws
            next_start = (cursor + words[i + 1][0]) if i + 1 < len(words) else (cursor + slot_dur)
            end_t = min(cursor + slot_dur, max(start_t + 0.12, min(cursor + we + 0.06, next_start)))
            clean_tok = token.replace("{", "").replace("}", "")
            styled_text = r"{\fscx114\fscy114\t(0,70,\fscx100\fscy100)}" + clean_tok
            events.append(
                f"Dialogue: 0,{fmt_ass_time(start_t)},{fmt_ass_time(end_t)},WordPop,,0,0,0,,{styled_text}"
            )
        cursor += slot_dur

    out_ass.parent.mkdir(parents=True, exist_ok=True)
    out_ass.write_text("\n".join(header + events) + "\n", encoding="utf-8")
    return out_ass


def build_youtube_package(episode: dict, scenes: list, final_video_path: Path, is_short: bool, output_dir: Path):
    title_base = str(episode.get("title") or TOPIC).strip()
    if len(title_base) > 85:
        title_base = title_base[:82].rstrip() + "..."
    yt_title = (f"{title_base} #Shorts" if is_short and "#shorts" not in title_base.lower() else title_base)[:100]

    scene_lines = []
    for idx, sc in enumerate(scenes, start=1):
        summary = str(sc.get("narration") or sc.get("text") or sc.get("summary") or "").strip()
        scene_lines.append(f"Beat {idx}: {summary}")

    tags = ["explained", "eli5", "2d animation", "storytime", "education", "visual storytelling"]
    if is_short:
        tags.extend(["shorts", "youtube shorts"])

    description = "\n\n".join([
        TOPIC,
        "\n".join(scene_lines),
        " ".join(f"#{t.replace(' ', '')}" for t in tags),
    ])

    pkg = {
        "project_id": PROJECT_ID,
        "studio": "2D Explainer Shorts Studio",
        "character": CHARACTER_NAME,
        "format": FORMAT_TYPE,
        "resolution": f"{TARGET_W}x{TARGET_H}",
        "video_file": str(final_video_path),
        "title": yt_title,
        "description": description,
        "tags": tags,
        "category_id": "27",
        "privacy_status": "private",
    }
    (output_dir / "youtube_publish_package.json").write_text(json.dumps(pkg, indent=2), encoding="utf-8")
    (output_dir / "youtube_publish_package.md").write_text(
        "\n".join([
            "# Ready-to-Publish 2D Explainer Metadata",
            "",
            f"- **Character:** `{CHARACTER_NAME}`",
            f"- **Format:** `{FORMAT_TYPE}` (`{pkg['resolution']}`)",
            f"- **Video File:** `{pkg['video_file']}`",
            "",
            "## Title",
            f"```\n{pkg['title']}\n```",
            "",
            "## Description",
            f"```\n{pkg['description']}\n```",
            "",
            "## Tags",
            f"`{', '.join(pkg['tags'])}`",
        ]),
        encoding="utf-8",
    )
    return pkg


try:
    DRIVE_PROJECT_DIR = DRIVE_WORKSPACE / "projects" / PROJECT_ID
    DRIVE_OUTPUT_DIR = DRIVE_PROJECT_DIR / "explainer_2d"
    BRAND_ASSETS_DIR = DRIVE_WORKSPACE / "brand_assets"
    BRAND_ASSETS_DIR.mkdir(parents=True, exist_ok=True)

    bible_file = BRAND_ASSETS_DIR / f"{re.sub(r'[^a-zA-Z0-9]+', '_', CHARACTER_NAME.lower())}_bible.json"
    bible_file.write_text(
        json.dumps(
            {
                "character_name": CHARACTER_NAME,
                "character_bible": CHARACTER_BIBLE,
                "updated_at": time.strftime("%Y-%m-%d %H:%M:%S"),
            },
            indent=2,
        ),
        encoding="utf-8",
    )

    if DRIVE_PROJECT_DIR.is_dir():
        hydrated = sync_tree(DRIVE_PROJECT_DIR, LOCAL_PROJECT_DIR)
        note(f"Hydrated {hydrated} new/updated file(s) from Drive into local SSD for resume-safe production.")

    write_log("\n\n===== 2D Explainer Studio run: " + time.strftime("%Y-%m-%d %H:%M:%S") + " =====\n", flush=True)
    note(f"Episode #{EPISODE_NUM:02d} | Project ID: {PROJECT_ID} | Character: {CHARACTER_NAME} | Scenes: {SCENE_COUNT}")
    note(f"Active Topic: {ACTIVE_TOPIC}")

    api_key = TOPIC_API_KEY

    if (ROOT / ".git").is_dir():
        run_logged("Fetch target branch " + CODE_BRANCH, ["git", "-C", str(ROOT), "fetch", "--depth=1", "origin", CODE_BRANCH])
        run_logged("Checkout target branch commit", ["git", "-C", str(ROOT), "checkout", "-B", CODE_BRANCH, "FETCH_HEAD"])
    else:
        if ROOT.exists():
            shutil.rmtree(ROOT)
        run_logged(
            "Clone target branch " + CODE_BRANCH,
            ["git", "clone", "--depth", "1", "--branch", CODE_BRANCH, "--single-branch", REPO_URL, str(ROOT)],
        )

    commit_sha = subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "--short", "HEAD"], text=True).strip()
    note(f"Active repository commit: {CODE_BRANCH} @ {commit_sha}")

    ensure_system_tools()
    ensure_python_deps(ROOT / "requirements.txt")

    plan_path = LOCAL_OUTPUT_DIR / "episode.json"
    force_plan = False
    if plan_path.is_file():
        try:
            prev_ep = json.loads(plan_path.read_text(encoding="utf-8"))
            previous_scene_count = len(prev_ep.get("scenes", []))
            if previous_scene_count != SCENE_COUNT:
                force_plan = True
                note(
                    f"Saved plan has {previous_scene_count} scenes but {SCENE_COUNT} are requested; "
                    "main.py will back it up before generating a replacement."
                )
            elif not episode_plan_matches_topic(prev_ep, ACTIVE_TOPIC):
                force_plan = True
                note(
                    "Saved episode.json does not match the active ledger topic. "
                    "main.py will back it up before generating the correct plan."
                )
            else:
                note("Saved plan matches the active topic and scene count; preserving it.")
        except Exception as plan_exc:
            force_plan = True
            note(f"Saved episode.json could not be validated ({type(plan_exc).__name__}); backing it up before replanning.")

    try:
        safe_step_count = int(STEP_COUNT)
    except Exception:
        safe_step_count = 3
    safe_step_count = max(1, min(6, safe_step_count))

    tts_patch_dir = Path("/content/.tts_patch")
    tts_patch_dir.mkdir(parents=True, exist_ok=True)
    (tts_patch_dir / "sitecustomize.py").write_text(
        """import asyncio
import os
import subprocess
import sys
import tempfile

_FALLBACK_VOICES = [
    "en-US-AndrewMultilingualNeural",
    "en-US-AriaNeural",
    "en-US-GuyNeural",
    "en-GB-RyanNeural",
]

try:
    import edge_tts
    from edge_tts.exceptions import NoAudioReceived

    _orig_init = edge_tts.Communicate.__init__
    _orig_save = edge_tts.Communicate.save
    _orig_stream = edge_tts.Communicate.stream

    def _tracked_init(self, text, voice="en-US-AndrewMultilingualNeural", *args, **kwargs):
        self._saved_init_text = text
        self._saved_init_voice = voice
        self._saved_init_args = args
        self._saved_init_kwargs = dict(kwargs)
        _orig_init(self, text, voice, *args, **kwargs)

    def _clone_communicate(src, voice_override=None):
        text = getattr(src, "_saved_init_text", None)
        if text is None:
            text = getattr(src, "_text", "")
        voice = voice_override or getattr(src, "_saved_init_voice", None) or getattr(src, "voice", None) or "en-US-AndrewMultilingualNeural"
        init_args = getattr(src, "_saved_init_args", ())
        init_kwargs = dict(getattr(src, "_saved_init_kwargs", {}))
        try:
            return edge_tts.Communicate(text, voice, *init_args, **init_kwargs)
        except TypeError:
            return edge_tts.Communicate(text, voice)

    def _guarded_stream(self, *args, **kwargs):
        state = getattr(self, "state", {})
        already_called = (
            getattr(self, "_tts_stream_claimed", False)
            or (isinstance(state, dict) and state.get("stream_was_called", False))
        )
        if already_called:
            fresh = _clone_communicate(self)
            fresh._tts_stream_claimed = True
            return _orig_stream(fresh, *args, **kwargs)
        self._tts_stream_claimed = True
        return _orig_stream(self, *args, **kwargs)

    async def _resilient_save(self, audio_fname, *args, **kwargs):
        base_voice = getattr(self, "_saved_init_voice", None) or getattr(self, "voice", None) or "en-US-AndrewMultilingualNeural"
        voices_to_try = [base_voice] + [v for v in _FALLBACK_VOICES if v != base_voice]
        last_exc = None

        for candidate_voice in voices_to_try:
            for attempt in range(3):
                try:
                    if os.path.exists(audio_fname):
                        try:
                            os.remove(audio_fname)
                        except OSError:
                            pass
                    runner = _clone_communicate(self, voice_override=candidate_voice)
                    await _orig_save(runner, audio_fname, *args, **kwargs)
                    if os.path.isfile(audio_fname) and os.path.getsize(audio_fname) > 256:
                        return
                    raise NoAudioReceived("Empty or truncated audio output")
                except NoAudioReceived as exc:
                    last_exc = exc
                    await asyncio.sleep(1.5 * (attempt + 1))
                except Exception as exc:
                    last_exc = exc
                    await asyncio.sleep(1.0 * (attempt + 1))
        if last_exc is not None:
            raise last_exc

    edge_tts.Communicate.__init__ = _tracked_init
    edge_tts.Communicate.stream = _guarded_stream
    edge_tts.Communicate.save = _resilient_save
except Exception:
    pass


def _is_valid_audio_file(path):
    if not path or not os.path.isfile(path) or os.path.getsize(path) < 256:
        return False
    probe = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries", "stream=codec_type", "-of", "csv=p=0", path],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
    )
    return probe.returncode == 0 and "audio" in (probe.stdout or "")


def _repair_bad_input_audio(mp3_path):
    if not mp3_path or not mp3_path.endswith(".mp3"):
        return
    if _is_valid_audio_file(mp3_path):
        return
    try:
        import edge_tts
        async def _regen():
            for v in _FALLBACK_VOICES:
                try:
                    comm = edge_tts.Communicate("Here is the next key point.", v)
                    await comm.save(mp3_path)
                    if _is_valid_audio_file(mp3_path):
                        return True
                except Exception:
                    await asyncio.sleep(1.0)
            return False
        if asyncio.run(_regen()):
            return
    except Exception:
        pass
    tmp_wav = tempfile.NamedTemporaryFile(suffix=".wav", delete=False).name
    try:
        subprocess.run(
            ["ffmpeg", "-y", "-f", "lavfi", "-i", "anullsrc=r=24000:cl=mono", "-t", "2.5", tmp_wav],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
        )
        subprocess.run(
            ["ffmpeg", "-y", "-i", tmp_wav, "-codec:a", "libmp3lame", "-q:a", "4", mp3_path],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
        )
    finally:
        if os.path.exists(tmp_wav):
            try:
                os.remove(tmp_wav)
            except OSError:
                pass


_orig_subprocess_run = subprocess.run

def _guarded_subprocess_run(args, *p_args, **p_kwargs):
    cmd = list(args) if isinstance(args, (list, tuple)) else []
    if cmd and os.path.basename(str(cmd[0])) == "ffmpeg" and "-i" in cmd:
        i_idx = cmd.index("-i")
        if i_idx + 1 < len(cmd):
            inp = str(cmd[i_idx + 1])
            if inp.endswith(".mp3") and not _is_valid_audio_file(inp):
                _repair_bad_input_audio(inp)
    return _orig_subprocess_run(args, *p_args, **p_kwargs)

subprocess.run = _guarded_subprocess_run
""",
        encoding="utf-8",
    )

    env = os.environ.copy()
    existing_pypath = env.get("PYTHONPATH", "")
    env.update({
        "PYTHONUNBUFFERED": "1",
        "PYTHONPATH": f"{tts_patch_dir}:{existing_pypath}" if existing_pypath else str(tts_patch_dir),
        "GEMINI_API_KEY": api_key,
        "MYSTERY_PLANNER_PROFILE": "pip_pattern_explainer",
        "ALLOW_PAID_IMAGE_API": "false",
        "MYSTERY_WHISPER_DEVICE": ACTIVE_WHISPER_DEVICE,
        "MYSTERY_WHISPER_MODEL": WHISPER_MODEL,
        "MYSTERY_NARRATOR_VOICE": VOICE,
        "MYSTERY_MUSIC_TRACK": str(LOCAL_OUTPUT_DIR / "no_music_selected.mp3"),
    })

    explainer_topic_prompt = str(ACTIVE_TOPIC).strip()
    prep_cmd = [
        sys.executable, str(ROOT / "main.py"),
        "--stage", "prepare",
        "--topic", explainer_topic_prompt,
        "--scene-count", str(SCENE_COUNT),
        "--output", str(LOCAL_OUTPUT_DIR),
        "--voice", VOICE,
        "--whisper-model", WHISPER_MODEL,
        "--whisper-device", ACTIVE_WHISPER_DEVICE,
        "--max-scenes-per-flow-run", str(MAX_SCENES_PER_FLOW_RUN),
    ]
    if force_plan:
        prep_cmd.append("--force-plan")
    run_logged(f"Stage 1/2: Prepare {SCENE_COUNT}-beat Pip & The Pattern explainer, narration & captions", prep_cmd, env=env)

    episode = json.loads(plan_path.read_text(encoding="utf-8"))
    scenes = episode.get("scenes", [])
    if len(scenes) != SCENE_COUNT:
        raise RuntimeError(f"Expected {SCENE_COUNT} scenes, found {len(scenes)}.")

    step_slots = compute_step_card_slots(scenes, safe_step_count, EPISODE_NUM, ACTIVE_TOPIC)
    for scene_idx, step_num in step_slots.items():
        card_png = FRAMES_DIR / f"scene_{scene_idx:02d}.png"
        if not card_png.is_file():
            step_headline = str(scenes[scene_idx - 1].get("title") or "").strip()
            render_handdrawn_step_card(step_num, step_headline, card_png, TARGET_W, TARGET_H)
            note(f"Auto-generated narration-synced card: Step {step_num} -> {card_png.name}")

    locked_md, prompt_map = build_locked_explainer_prompts(scenes, step_slots, LOCAL_OUTPUT_DIR)

    missing_scenes = []
    scene_assets = {}
    for idx in range(1, SCENE_COUNT + 1):
        kind, asset_path = find_scene_asset(CLIPS_DIR, FRAMES_DIR, idx)
        if kind is None and idx in prompt_map and api_key:
            if ASSET_MODE == "gemini_image_api_if_missing":
                target_png = FRAMES_DIR / f"scene_{idx:02d}.png"
                if try_generate_still_with_imagen(api_key, IMAGE_MODEL, prompt_map[idx], target_png, ASPECT_STR):
                    kind, asset_path = ("image", target_png)
            elif ASSET_MODE == "veo_api_if_missing":
                target_mp4 = CLIPS_DIR / f"scene_{idx:02d}.mp4"
                if try_generate_clip_with_veo(api_key, VEO_MODEL, prompt_map[idx], target_mp4, ASPECT_STR):
                    kind, asset_path = ("video", target_mp4)

        if kind is None:
            missing_scenes.append(f"scene_{idx:02d}")
        else:
            scene_assets[idx] = (kind, asset_path)

    record_ledger_entry(LEDGER, EPISODE_NUM, PROJECT_ID, ACTIVE_TOPIC, "awaiting_visuals")
    save_ledger(LEDGER_PATH, LEDGER)
    if missing_scenes:
        note(
            f"Stage 1 ready! {len(step_slots)} narration-synced Step Card(s) auto-generated in frames/. "
            f"Generate the remaining {len(missing_scenes)} visual beat(s) using the locked Character Bible prompts below "
            f"(save as scene_XX.mp4 in flow_clips/ OR still images scene_XX.png in frames/ for Ken Burns animation), "
            f"then re-run with RESUME_PROJECT_ID='{PROJECT_ID}'."
        )
        display(
            Markdown(
                f"## Stage 1 Complete — Locked Character Prompts Ready\n\n"
                f"- **Project ID:** `{PROJECT_ID}`\n\n"
                f"- **Locked Character:** `{CHARACTER_NAME}` (saved to `brand_assets/{bible_file.name}`)\n\n"
                f"- **Auto-Generated Step Cards (Narration-Synced):** `{len(step_slots)}` (`{', '.join(f'scene_{k:02d}.png (Step {v})' for k, v in step_slots.items())}`)\n\n"
                f"- **Upload Video Clips To:** `{DRIVE_OUTPUT_DIR / 'flow_clips'}` (`scene_01.mp4` …)\n\n"
                f"- **OR Upload Still Illustrations To:** `{DRIVE_OUTPUT_DIR / 'frames'}` (`scene_01.png` … for automatic Ken Burns pan/zoom)\n\n"
                f"- **Remaining Scenes Needed ({len(missing_scenes)}):** `{', '.join(missing_scenes)}`"
            )
        )
        display(
            HTML(
                f"<details open style='{DETAILS_STYLE}'>"
                f"<summary style='font-weight:bold; cursor:pointer;'>🎨 Locked Character & Art-Style Prompts (Copy-Paste to Google Flow)</summary>"
                f"<pre style='{PRE_BLOCK_STYLE}'>{html.escape(locked_md)}</pre>"
                f"</details>"
            )
        )
    else:
        encoder = detect_video_encoder(ENCODER_MODE)
        sfx_vol = max(0.0, min(1.0, float(FLOW_SFX_VOLUME)))
        note(f"All {SCENE_COUNT} scenes ready. Rendering 2D Explainer ({FORMAT_TYPE}) with {encoder}...")

        narration_manifest_path = LOCAL_OUTPUT_DIR / "audio" / "narration_manifest.json"
        narration_data = (
            json.loads(narration_manifest_path.read_text(encoding="utf-8"))
            if narration_manifest_path.is_file()
            else {}
        )
        scene_audio_items = narration_data.get("scenes") or narration_data.get("segments") or []

        norm_clips = []
        sfx_clips = []
        audio_tracks = []
        total_dur = 0.0

        for idx in range(1, SCENE_COUNT + 1):
            kind, src_path = scene_assets[idx]
            audio_path = None
            if idx - 1 < len(scene_audio_items) and isinstance(scene_audio_items[idx - 1], dict):
                raw_aud = scene_audio_items[idx - 1].get("audio_path") or scene_audio_items[idx - 1].get("file")
                if raw_aud:
                    cand = Path(raw_aud)
                    if not cand.is_absolute():
                        cand = (
                            (LOCAL_OUTPUT_DIR / raw_aud)
                            if (LOCAL_OUTPUT_DIR / raw_aud).exists()
                            else (LOCAL_OUTPUT_DIR / "audio" / Path(raw_aud).name)
                        )
                    if cand.is_file():
                        audio_path = cand

            if audio_path is None:
                for cand in sorted((LOCAL_OUTPUT_DIR / "audio").glob(f"*scene*{idx:02d}*")):
                    if cand.suffix.lower() in (".mp3", ".wav", ".m4a"):
                        audio_path = cand
                        break

            clean_aud = ASSEMBLED_DIR / f"clean_narration_{idx:02d}.wav"
            if audio_path is not None:
                af_filters = ["aformat=sample_fmts=s16:sample_rates=44100:channel_layouts=stereo"]
                if VOICE_WARMTH_EQ:
                    af_filters.extend([
                        "equalizer=f=150:t=q:w=1.2:g=2.2",
                        "equalizer=f=3400:t=q:w=1.5:g=1.4",
                        "acompressor=threshold=-18dB:ratio=2.5:attack=12:release=120",
                    ])
                run_cmd(["ffmpeg", "-y", "-i", str(audio_path), "-af", ",".join(af_filters), str(clean_aud)])
                target_dur = max(1.6, round(probe_media_duration(clean_aud), 3))
            else:
                target_dur = 2.2

            total_dur += target_dur
            norm_out = ASSEMBLED_DIR / f"scene_{idx:02d}_norm.mp4"
            sfx_out = ASSEMBLED_DIR / f"scene_{idx:02d}_sfx.wav"

            enc_args = (
                ["-c:v", "h264_nvenc", "-rc", "vbr", "-cq", "20", "-pix_fmt", "yuv420p"]
                if encoder == "h264_nvenc"
                else ["-c:v", "libx264", "-crf", "19", "-preset", "fast", "-pix_fmt", "yuv420p"]
            )

            if kind == "image":
                still_input = src_path
                if src_path.suffix.lower() in (".webp", ".gif"):
                    converted_png = ASSEMBLED_DIR / f"still_input_{idx:02d}.png"
                    run_cmd(["ffmpeg", "-y", "-i", str(src_path), "-frames:v", "1", str(converted_png)])
                    still_input = converted_png

                frames_n = max(30, int(round(target_dur * 30)))
                if idx % 2 == 1:
                    zoom_expr = "min(zoom+0.0014,1.12)"
                else:
                    zoom_expr = "if(lte(on,1),1.12,max(1.00,zoom-0.0014))"
                vf_img = (
                    f"scale={TARGET_W * 2}:{TARGET_H * 2}:force_original_aspect_ratio=increase,"
                    f"crop={TARGET_W * 2}:{TARGET_H * 2},"
                    f"zoompan=z='{zoom_expr}':x='iw/2-(iw/zoom/2)':y='ih/2-(ih/zoom/2)':"
                    f"d={frames_n}:s={TARGET_W}x{TARGET_H}:fps=30,"
                    f"format=yuv420p"
                )
                run_cmd([
                    "ffmpeg", "-y", "-framerate", "30", "-loop", "1", "-i", str(still_input),
                    "-t", f"{target_dur:.3f}",
                    "-vf", vf_img,
                    "-an", *enc_args, str(norm_out),
                ])
            else:
                clip_dur = probe_media_duration(src_path)
                if clip_dur < target_dur:
                    pts_factor = round(target_dur / clip_dur, 5)
                    vf_vid = [
                        f"setpts={pts_factor}*PTS",
                        f"scale={TARGET_W}:{TARGET_H}:force_original_aspect_ratio=increase",
                        f"crop={TARGET_W}:{TARGET_H}",
                        "fps=30", "format=yuv420p",
                    ]
                else:
                    vf_vid = [
                        "setpts=PTS-STARTPTS",
                        f"scale={TARGET_W}:{TARGET_H}:force_original_aspect_ratio=increase",
                        f"crop={TARGET_W}:{TARGET_H}",
                        "fps=30", "format=yuv420p",
                    ]
                run_cmd([
                    "ffmpeg", "-y", "-i", str(src_path),
                    "-t", f"{target_dur:.3f}",
                    "-vf", ",".join(vf_vid),
                    "-an", *enc_args, str(norm_out),
                ])

            if kind == "video" and sfx_vol > 0 and has_audio_stream(src_path):
                run_cmd([
                    "ffmpeg", "-y", "-i", str(src_path),
                    "-t", f"{target_dur:.3f}",
                    "-af", (
                        f"aformat=sample_fmts=s16:sample_rates=44100:channel_layouts=stereo,"
                        f"apad=whole_dur={target_dur:.3f},atrim=duration={target_dur:.3f},"
                        f"volume={sfx_vol}"
                    ),
                    str(sfx_out),
                ])
            else:
                run_cmd([
                    "ffmpeg", "-y", "-f", "lavfi", "-i", "anullsrc=r=44100:cl=stereo",
                    "-t", f"{target_dur:.3f}", str(sfx_out),
                ])

            norm_clips.append(norm_out)
            sfx_clips.append(sfx_out)
            if clean_aud.is_file():
                audio_tracks.append((clean_aud, target_dur))
            note(f"Rendered beat {idx:02d} ({kind}, {target_dur:.2f}s)")

        concat_list = ASSEMBLED_DIR / "concat_list.txt"
        concat_list.write_text("\n".join(f"file '{p.as_posix()}'" for p in norm_clips) + "\n", encoding="utf-8")
        merged_video = ASSEMBLED_DIR / "merged_visuals.mp4"
        run_cmd(["ffmpeg", "-y", "-f", "concat", "-safe", "0", "-i", str(concat_list), "-c:v", "copy", str(merged_video)])

        sfx_list = ASSEMBLED_DIR / "sfx_concat.txt"
        sfx_list.write_text("\n".join(f"file '{s.as_posix()}'" for s in sfx_clips) + "\n", encoding="utf-8")
        merged_sfx = ASSEMBLED_DIR / "merged_sfx.wav"
        run_cmd(["ffmpeg", "-y", "-f", "concat", "-safe", "0", "-i", str(sfx_list), "-c:a", "pcm_s16le", str(merged_sfx)])

        merged_narration = ASSEMBLED_DIR / "merged_narration.wav"
        if audio_tracks:
            nar_list = ASSEMBLED_DIR / "narration_concat.txt"
            seg_wavs = []
            for idx, (ap, td) in enumerate(audio_tracks, start=1):
                seg_w = ASSEMBLED_DIR / f"nar_{idx:02d}.wav"
                run_cmd(["ffmpeg", "-y", "-i", str(ap), "-af", f"apad=whole_dur={td:.3f},atrim=duration={td:.3f}", str(seg_w)])
                seg_wavs.append(seg_w)
            nar_list.write_text("\n".join(f"file '{w.as_posix()}'" for w in seg_wavs) + "\n", encoding="utf-8")
            run_cmd(["ffmpeg", "-y", "-f", "concat", "-safe", "0", "-i", str(nar_list), "-c:a", "pcm_s16le", str(merged_narration)])

        mixed_audio = ASSEMBLED_DIR / "final_mix.wav"
        if merged_narration.is_file() and merged_sfx.is_file():
            run_cmd([
                "ffmpeg", "-y", "-i", str(merged_narration), "-i", str(merged_sfx),
                "-filter_complex", "[0:a]volume=1.05[voice];[1:a]volume=1.0[sfx];[voice][sfx]amix=inputs=2:duration=first[aout]",
                "-map", "[aout]", "-c:a", "pcm_s16le", str(mixed_audio),
            ])
        elif merged_narration.is_file():
            mixed_audio = merged_narration

        vf_final = []
        if CAPTION_STYLE == "word_pop_center" and audio_tracks:
            ass_path = ASSEMBLED_DIR / "word_pop_captions.ass"
            generate_word_pop_ass(audio_tracks, WHISPER_MODEL, ACTIVE_WHISPER_DEVICE, ass_path, TARGET_W, TARGET_H)
            escaped_ass = str(ass_path).replace("\\", "/").replace(":", "\\:")
            vf_final.append(f"ass='{escaped_ass}'")
        elif CAPTION_STYLE == "lower_third_clean":
            srt_path = LOCAL_OUTPUT_DIR / "captions" / "subtitles.srt"
            if srt_path.is_file() and srt_path.stat().st_size > 0:
                margin_v = 260 if IS_SHORT else 60
                font_sz = 36 if IS_SHORT else 42
                escaped_srt = str(srt_path).replace("\\", "/").replace(":", "\\:")
                vf_final.append(
                    f"subtitles='{escaped_srt}':force_style='PlayResX={TARGET_W},PlayResY={TARGET_H},"
                    f"FontName=DejaVu Sans,FontSize={font_sz},Bold=0,PrimaryColour=&H00E8ECEF,"
                    f"OutlineColour=&H88101418,BorderStyle=1,Outline=1.2,Shadow=1,Alignment=2,MarginV={margin_v}'"
                )

        final_video_path = ASSEMBLED_DIR / FINAL_VIDEO_NAME
        final_cmd = ["ffmpeg", "-y", "-i", str(merged_video)]
        if mixed_audio.is_file():
            final_cmd.extend(["-i", str(mixed_audio), "-map", "0:v:0", "-map", "1:a:0", "-c:a", "aac", "-b:a", "192k"])
        else:
            final_cmd.extend(["-map", "0:v:0", "-an"])
        if vf_final:
            final_cmd.extend(["-vf", ",".join(vf_final)])

        enc_final = (
            ["-c:v", "h264_nvenc", "-rc", "vbr", "-cq", "19", "-pix_fmt", "yuv420p", "-movflags", "+faststart"]
            if encoder == "h264_nvenc"
            else ["-c:v", "libx264", "-crf", "18", "-preset", "medium", "-pix_fmt", "yuv420p", "-movflags", "+faststart"]
        )
        final_cmd.extend([*enc_final, str(final_video_path)])
        run_logged(f"Stage 2/2: Render final 2D Explainer ({FINAL_VIDEO_NAME})", final_cmd)

        final_dur = probe_media_duration(final_video_path)
        yt_pkg = build_youtube_package(episode, scenes, final_video_path, IS_SHORT, ASSEMBLED_DIR)
        record_ledger_entry(LEDGER, EPISODE_NUM, PROJECT_ID, ACTIVE_TOPIC, "completed")
        save_ledger(LEDGER_PATH, LEDGER)
        note(f"SUCCESS: Episode #{EPISODE_NUM:02d} created at {final_video_path} ({final_dur:.2f}s)")

        display(
            Markdown(
                f"## Ready-to-Publish 2D Explainer Package ({FORMAT_TYPE})\n\n"
                f"- **Video File (Drive):** `{DRIVE_OUTPUT_DIR / 'assembled' / FINAL_VIDEO_NAME}`\n"
                f"- **Duration:** `{final_dur:.2f}s` across `{SCENE_COUNT}` beats\n"
                f"- **Character Locked:** `{CHARACTER_NAME}`\n"
                f"- **Title:** `{yt_pkg['title']}`\n"
                f"- **Metadata Files:** `youtube_publish_package.json` & `youtube_publish_package.md`"
            )
        )

except Exception as exc:
    error = f"FAILED: {type(exc).__name__}: {exc}"
    print("\n" + error, flush=True)
    try:
        write_log("\n" + error + "\n" + traceback.format_exc() + "\n", flush=True)
        tail = LOG_PATH.read_text(encoding="utf-8", errors="replace").splitlines()[-35:]
        print("\nLast log lines:", flush=True)
        print("\n".join(tail), flush=True)
    except Exception:
        print(traceback.format_exc(), flush=True)

finally:
    if DRIVE_PROJECT_DIR is not None and LOCAL_PROJECT_DIR.is_dir():
        try:
            print(f"\nSyncing 2D Explainer artifacts to Google Drive ({DRIVE_PROJECT_DIR})…", flush=True)
            synced_count = sync_tree(LOCAL_PROJECT_DIR, DRIVE_PROJECT_DIR)
            print(f"Drive sync complete ({synced_count} file(s) copied): {DRIVE_OUTPUT_DIR}", flush=True)
        except Exception as sync_exc:
            print(f"WARNING: Failed to sync to Google Drive: {sync_exc}", flush=True)
    try:
        flush_log()
        log_stream.close()
    except Exception:
        pass
